In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

In [3]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/post_cell_9.pickle

trying: ['x']
me:  14
trying: ['df_cell6']


me:  18
trying: ['pd']
me:  0
trying: ['null_rate']
me:  4
trying: ['ratings_ages']
me:  18
trying: ['i']
me:  4
trying: ['warnings']
me:  1
trying: ['df']
me:  3
trying: ['r']
me:  14
trying: ['data']
me:  20
trying: ['Path']
me:  0
trying: ['y']
me:  14
trying: ['np']
me:  0
trying: ['orig_output']
me:  15
trying: ['mf_ratio']
me:  14
trying: ['BENCHMARKS_TO_PATHS']
me:  0
trying: ['i_2']
me:  16
trying: ['benchmark_name']
me:  2
trying: ['i_1']
me:  16
trying: ['factor']
me:  2
Declaring variable pd
Declaring variable Path
Declaring variable np
Declaring variable BENCHMARKS_TO_PATHS
Declaring variable warnings
Declaring variable benchmark_name
Declaring variable factor
Declaring variable df
Declaring variable null_rate
Declaring variable i
Declaring variable x
Declaring variable r
Declaring variable y
Declaring variable mf_ratio
Declaring variable orig_output
Declaring variable i_2
Declaring variable i_1
Declaring variable df_cell6
Declaring variable ratings_ages
Declaring variable 

In [4]:
%%RecordEvent
%%time
### cell 10 ###

# Rewritten for cudf (avoid unstack and axis-1/div CPU fallbacks)

# 1. Top 11 countries by count (value_counts is GPU)
country_order = df_cell6['first_country'].value_counts().head(11).index

# 2. Compute the (country, type) counts in long form on GPU
counts = (
    df_cell6
    .groupby(['first_country', 'type'])
    .size()
    .reset_index(name='count')
)

# 3. Pivot to wide form WITHOUT unstack
movies = counts[counts['type'] == 'Movie'][['first_country', 'count']]
movies = movies.rename(columns={'count': 'Movie'})

tv = counts[counts['type'] == 'TV Show'][['first_country', 'count']]
tv = tv.rename(columns={'count': 'TV Show'})

# 4. Merge on GPU and reindex
data_q2q3 = movies.merge(tv, on='first_country', how='outer').fillna(0)
data_q2q3 = data_q2q3.set_index('first_country').loc[country_order]

# 5. Compute row sum with GPU addition
data_q2q3['sum'] = data_q2q3['Movie'] + data_q2q3['TV Show']

# 6. Compute ratios elementwise (GPU) and sort
#    Avoid DataFrame.div (CPU fallback) and axis=1 sum
data_q2q3_ratio = data_q2q3[['Movie', 'TV Show']].copy()
# elementwise division stays on GPU
data_q2q3_ratio['Movie']   = data_q2q3_ratio['Movie']   / data_q2q3['sum']
data_q2q3_ratio['TV Show'] = data_q2q3_ratio['TV Show'] / data_q2q3['sum']

# sort_values is GPU
data_q2q3_ratio = data_q2q3_ratio.sort_values('Movie', ascending=True)

CPU times: user 778 ms, sys: 255 ms, total: 1.03 s
Wall time: 1.02 s


In [5]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/rewritten/o4_mini_high_small/checkpoints/post_cell_10_try_1.pickle

migration speed (bps): 886929420.8391325


---------------------------
variables to migrate:
np 72
tv 1623
y 28
data_q2q3_ratio 284
null_rate 32
BENCHMARKS_TO_PATHS 2272
pd 72
Path 904
warnings 72
i 60
r 40
mf_ratio 28
df 9352096
i_2 53
i_1 53
counts 4050
country_order 1005
x 40
data_q2q3 372
df_cell6 595448600
data 179
orig_output 16
factor 28
benchmark_name 75
ratings_ages 640
movies 2171
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/rewritten/o4_mini_high_small/checkpoints/post_cell_10_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables ['BENCHMARKS_TO_PATHS', 'Path']
Active variables ['df', 'df_cell6']
Intermediate variables ['factor', 'benchmark_name']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['df_cell6']
Active variables []
Intermediate variables ['i', 'null_rate']
Future variables []
Modified dataframes
======= Cell 2 =======
Input variables ['df_cell6']
Active variables ['df_cell6']
Intermediate variables []
Future variables []
Modified dataframes
  df_cell6
    Input columns: set()
    Changed columns: {'director', 'country', 'description', 'listed_in', 'cast', 'show_id', 'release_year', 'title', 'duration', 'rating', 'date_added', 'type'}
    Created columns: set()
    Deleted columns: set()
======= Cell 3 =======
Input variables ['df_cell6']
Active variables []
Intermediate variables []
Future variables ['df_cell6']
Modified dataframes
======= Cell 4 =======
Input variables ['df_cell6']
Active variables []
Intermediate variables []
Fu

In [7]:

with open("/scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/opt_cell_exec_info_10_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[10], f)


In [8]:
opt_output = Out.get(4)

In [9]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/post_cell_10.pickle

import numpy as np
if os.getenv("USE_GPU") == "True":
    import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
if os.getenv("USE_GPU") == "True":
    is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
    is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
else:
    is_orig_output_array = isinstance(orig_output, np.ndarray)
    is_opt_output_array = isinstance(opt_output, np.ndarray)

is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['np']
me:  0
trying: ['Path']
me:  0
trying: ['y']
me:  14
trying: ['df']
me:  3
trying: ['pd']
me:  0
trying: ['warnings']
me:  1
trying: ['null_rate']
me:  4
trying: ['r']
me:  14
trying: ['BENCHMARKS_TO_PATHS']
me:  0
trying: ['data_q2q3_ratio']
me:  22
trying: ['mf_ratio']
me:  14
trying: ['i_2']
me:  16
trying: ['i']
me:  4
trying: ['i_1']
me:  16
trying: ['data_q2q3']
me:  22
trying: ['x']
me:  14
trying: ['df_cell6']


me:  18
trying: ['data']


me:  20
trying: ['orig_output']
me:  15
trying: ['factor']
me:  2
trying: ['benchmark_name']
me:  2
trying: ['ratings_ages']
me:  18
trying: ['country_order']
me:  22
Declaring variable np
Declaring variable Path
Declaring variable pd
Declaring variable BENCHMARKS_TO_PATHS
Declaring variable warnings
Declaring variable factor
Declaring variable benchmark_name
Declaring variable df
Declaring variable null_rate
Declaring variable i
Declaring variable y
Declaring variable r
Declaring variable mf_ratio
Declaring variable x
Declaring variable orig_output
Declaring variable i_2
Declaring variable i_1
Declaring variable df_cell6
Declaring variable ratings_ages
Declaring variable data
Declaring variable data_q2q3_ratio
Declaring variable data_q2q3
Declaring variable country_order
